# glass-0-1-2-3_vs_4-5-6: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('14_glass-0-1-2-3_vs_4-5-6_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,89.009,85.220,89.810,82.364,96.932,89.648,94.279,98.246,87.115
smote,93.842,89.901,89.013,92.000,96.013,94.007,94.719,98.511,91.871
bsmote,94.187,90.026,88.399,93.000,95.701,94.350,94.470,98.539,92.106
ros,93.514,89.770,89.622,91.091,96.326,93.708,94.955,98.544,91.642
adasyn,94.187,90.026,88.399,93.000,95.701,94.350,93.834,98.410,92.106
casb,93.514,89.770,89.622,91.091,96.326,93.708,94.912,98.515,91.642
casb_relaxed,94.554,90.873,89.824,93.091,96.326,94.708,95.088,98.574,92.713
casb_edited,93.028,87.319,84.171,93.000,93.551,93.276,93.760,98.325,90.173
casb_ros,94.019,90.294,89.713,92.091,96.326,94.208,94.561,98.396,92.157


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.351,83.398,93.278,76.364,98.163,87.263,94.669,97.801,84.874
smote,89.060,83.964,86.398,84.182,95.085,89.634,93.210,97.643,86.512
bsmote,91.661,87.361,88.148,88.182,95.710,91.946,93.624,97.751,89.511
ros,90.536,86.757,89.586,85.182,96.619,90.901,95.131,97.889,88.646
adasyn,91.863,87.189,86.841,89.182,95.066,92.124,95.851,98.104,89.526
casb,90.040,85.500,87.678,85.273,95.701,90.487,93.112,97.702,87.770
casb_relaxed,89.313,84.538,86.860,84.182,95.388,89.785,93.634,97.601,86.926
casb_edited,88.391,82.279,83.037,84.182,93.551,88.866,89.690,96.920,85.335
casb_ros,88.711,84.279,87.505,82.273,96.004,89.138,93.653,97.648,86.495


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,91.531,87.030,87.145,88.000,95.701,91.850,96.047,98.580,89.280
smote,94.384,90.372,88.568,93.000,96.013,94.507,96.513,98.732,92.378
bsmote,93.925,89.259,86.708,93.000,95.085,94.043,95.806,98.441,91.592
ros,91.140,86.659,86.749,87.000,95.710,91.355,95.774,98.091,88.900
adasyn,92.502,87.904,86.838,90.000,95.398,92.699,95.170,98.017,90.203
casb,92.147,87.081,85.552,90.000,94.782,92.391,95.954,98.414,89.614
casb_relaxed,93.941,89.272,86.526,93.000,95.085,94.043,96.240,98.671,91.607
casb_edited,93.833,87.740,82.749,95.000,92.945,93.973,94.723,98.059,90.786
casb_ros,93.654,89.168,87.002,92.091,95.398,93.744,95.909,98.231,91.411


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,87.666,83.077,87.348,80.727,96.307,88.517,95.159,97.712,85.372
smote,89.743,84.562,85.421,85.364,95.095,90.229,94.628,97.594,87.152
bsmote,90.741,85.562,85.444,87.364,94.792,91.078,94.631,97.564,88.151
ros,88.944,83.851,85.887,83.636,95.398,89.517,94.516,97.595,86.398
adasyn,90.186,84.868,85.057,86.364,94.792,90.578,94.843,97.710,87.527
casb,87.958,82.913,86.111,81.727,95.701,88.714,95.007,97.803,85.436
casb_relaxed,89.009,84.116,86.361,83.545,95.701,89.623,94.571,97.589,86.563
casb_edited,88.550,82.374,82.383,84.364,93.570,88.967,92.496,96.840,85.462
casb_ros,89.101,84.247,86.528,83.636,95.701,89.669,95.244,97.955,86.674


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,88.639,84.681,89.395,81.864,96.776,89.320,95.038,98.085,86.660
smote,91.757,87.200,87.350,88.636,95.552,92.094,94.768,98.120,89.478
bsmote,92.628,88.052,87.175,90.386,95.322,92.854,94.633,98.074,90.340
ros,91.034,86.759,87.961,86.727,96.013,91.370,95.094,98.030,88.897
adasyn,92.184,87.497,86.784,89.636,95.239,92.438,94.925,98.060,89.841
casb,90.915,86.316,87.241,87.023,95.627,91.325,94.746,98.108,88.615
casb_relaxed,91.704,87.200,87.393,88.455,95.625,92.040,94.883,98.109,89.452
casb_edited,90.950,84.928,83.085,89.136,93.404,91.270,92.667,97.536,87.939
casb_ros,91.371,86.997,87.687,87.523,95.857,91.690,94.842,98.058,89.184


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,94.554,bsmote,94.187,0.367,WIN
1,f1,casb_relaxed,90.873,bsmote,90.026,0.847,WIN
2,precision,casb_relaxed,89.824,ros,89.622,0.202,WIN
3,recall,casb_relaxed,93.091,bsmote,93.000,0.091,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,90.040,adasyn,91.863,-1.823,LOSS
1,f1,casb,85.500,bsmote,87.361,-1.861,LOSS
2,precision,casb,87.678,ros,89.586,-1.908,LOSS
3,recall,casb,85.273,adasyn,89.182,-3.909,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,93.941,smote,94.384,-0.443,LOSS
1,f1,casb_relaxed,89.272,smote,90.372,-1.099,LOSS
2,precision,casb_ros,87.002,smote,88.568,-1.565,LOSS
3,recall,casb_edited,95.000,smote,93.000,2.000,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,91.704,bsmote,92.628,-0.924,LOSS
1,f1,casb_relaxed,87.200,bsmote,88.052,-0.852,LOSS
2,precision,casb_ros,87.687,ros,87.961,-0.274,LOSS
3,recall,casb_edited,89.136,bsmote,90.386,-1.250,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.890289,0.857143,0.900000,0.818182,0.968750,0.893466,0.975069,0.991477,0.873716,None
1,42,1,svm,smote,0.50,0.984251,0.956522,0.916667,1.000000,0.968750,0.984375,0.984160,0.994318,0.970386,None
2,42,1,svm,bsmote,0.75,0.968246,0.916667,0.846154,1.000000,0.937500,0.968750,0.984160,0.994318,0.942456,None
3,42,1,svm,ros,0.50,0.938446,0.909091,0.909091,0.909091,0.968750,0.938920,0.984160,0.994318,0.923769,None
4,42,1,svm,adasyn,0.75,0.968246,0.916667,0.846154,1.000000,0.937500,0.968750,0.992424,0.997159,0.942456,None
5,42,1,svm,casb,1.00,0.984251,0.956522,0.916667,1.000000,0.968750,0.984375,0.984160,0.994318,0.970386,None
6,42,1,svm,casb_relaxed,0.75,0.938446,0.909091,0.909091,0.909091,0.968750,0.938920,0.984160,0.994318,0.923769,None
7,42,1,svm,casb_edited,0.75,0.968246,0.916667,0.846154,1.000000,0.937500,0.968750,0.984160,0.994318,0.942456,None
8,42,1,svm,casb_ros,1.00,0.984251,0.956522,0.916667,1.000000,0.968750,0.984375,0.984160,0.994318,0.970386,None
9,42,1,knn,original,0.50,0.904534,0.900000,1.000000,0.818182,1.000000,0.909091,0.965368,0.985795,0.902267,None
